In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from miner.miner import *
import time

miner_dir    = r"C:\Users\Amogh Kukreja\AppData\Roaming\Python\Python313\site-packages\miner"
matrices_dir = r"D:\School\IITD\General\GBM_new\miner_output\output_matrices"
jsons_dir    = r"D:\School\IITD\General\GBM_new\miner_output\output_jsons"
data_dir     = r"D:\School\IITD\General\GBM_new\miner_input_data"

In [2]:
# Loading inputs:


# causal_results: 2072 * 13 — somatic mutation causal flows
# Each row is one (mutation, TF, regulon) triple with statistics
# Columns: Mutation, Regulator, Regulon, MutationRegulatorEdge, etc.
causal_results = pd.read_csv(os.path.join(matrices_dir, "filteredCausalResults.csv"), index_col=0)

# Remove the 'R-' prefix so that causal_results matches eigengene index
causal_results['Regulon'] = causal_results['Regulon'].astype(str).str.replace('R-', '', regex=False)

# exp_data: 10394 genes (symbols) x 550 patients — z-scored expression
exp_data = pd.read_csv(os.path.join(data_dir, "combined_zscored.csv"), index_col=0)

# eigengenes: 4504 regulons x 547 patients — PC1 score per regulon per patient
eigengenes = pd.read_csv(os.path.join(matrices_dir, "eigengenes.csv"), index_col=0)

# transcriptional_states: state_id -> list of patient IDs
transcriptional_states = read_json(os.path.join(jsons_dir, "transcriptional_states.json"))

# ensembl_to_symbol mapping from miner identifier_mappings.txt
idmap = pd.read_csv(os.path.join(miner_dir, "data", "identifier_mappings.txt"), sep="\t")
gene_name_map = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))
symbol_to_ensembl = {v: k for k, v in ensembl_to_symbol.items()}

# TFBSDB + ChEA binding site databases
tfbsdb_tf_to_genes = pickle.load(open(os.path.join(miner_dir, "data", "network_dictionaries", "tfbsdb_tf_to_genes.pkl"), 'rb'))
tfbsdb2_tf_to_genes = pickle.load(open(os.path.join(miner_dir, "data", "network_dictionaries", "tfbsdb2_tf_to_genes.pkl"), 'rb'))

# THis one is experimentally verified, rest are computational predictions
chea_seq = pickle.load(open(os.path.join(miner_dir, "data", "network_dictionaries", "chea_seq.pkl"), 'rb'))

print("causal_results:", causal_results.shape)
print("exp_data:", exp_data.shape)
print("eigengenes:", eigengenes.shape)
print("transcriptional_states:", len(transcriptional_states), "states")
print("ensembl_to_symbol mappings:", len(ensembl_to_symbol))

causal_results: (2072, 13)
exp_data: (10394, 550)
eigengenes: (4504, 547)
transcriptional_states: 23 states
ensembl_to_symbol mappings: 20055


In [ ]:
# Prepare TF-TF network inputs: extract 254 UNIQUE implicated TFs from causal flows

causal_tfs_symbols = causal_results['Regulator'].unique().tolist()
print(f"TFs from causal results: {len(causal_tfs_symbols)}")

# Convert gene symbol to Ensembl for TFBSDB lookup
causal_tfs_ensembl = [symbol_to_ensembl.get(tf) for tf in causal_tfs_symbols]
causal_tfs_ensembl = [t for t in causal_tfs_ensembl if t is not None]
print(f"TFs mapped to Ensembl: {len(causal_tfs_ensembl)}")

# Combine TFBSDB v1 + v2 + ChEA: TF (Ensembl ID) -> genes it binds in any one of the databases
combined_tf_to_genes = {}
for tf in causal_tfs_ensembl:
    genes = set()
    if tf in tfbsdb_tf_to_genes:
        genes.update(tfbsdb_tf_to_genes[tf])
    if tf in tfbsdb2_tf_to_genes:
        genes.update(tfbsdb2_tf_to_genes[tf])
    if tf in chea_seq:
        genes.update(chea_seq[tf])
    combined_tf_to_genes[tf] = list(genes)
print(f"TFs with binding site data: {len(combined_tf_to_genes)}")

# Expression matrix for causal TFs only (254 x 550)
tf_exp = exp_data.reindex(causal_tfs_symbols).dropna()
print(f"TFs with expression data: {tf_exp.shape[0]}")

In [ ]:
# Building TF-TF network using LASSO regression:

# For each target TF, look through other TFs and see if they bind to the target's promoter. These COULD regulate target
# Fit model for this across all patients and penalise complexity so that only TFs with MEANINGFUL effect are chosen (other coeffs become 0)
# All non-0 coefficients are results (>0: activates, <0: represses)
# Directed edge, e.g: E2F1 → SOX2 means that E2F1 expression activates SOX2 expression

# Hence A regulates B if 1) A has a binding site in B's promoter and 2) expression of A meaningfully affects that of B

start = time.time()

tf_tf_edges = []
# List of dicts; each dict is a directed edge
# E.g: 

# {'Source': 'E2F1',    # TF doing the regulating, 
# 'Target': 'SOX2',    # TF being regulated  ,
# 'Edge':   1,         # +1 = activates, -1 = represses
# 'Coef':   0.342}     # LASSO coefficient magnitude


print(f"Building TF-TF network for {tf_exp.shape[0]} TFs...")

# Iterates over every one of the 254 TFs, taking each as its target 
for i, target_tf in enumerate(tf_exp.index):
    if i % 25 == 0:

        # Display progress every 25 TFs
        print(f"{i}/{tf_exp.shape[0]}")
        
    # Ensembl ID corresponding to the target TF; if it doesn't exist, simply bypass
    target_ensembl = symbol_to_ensembl.get(target_tf)
    if target_ensembl is None:
        continue

    # Find TFs with binding sites for this target TF
    predictor_tfs = []

    # combined_tf_to_genes is a dict with {TF Ensembl ID (str) : genes mapped to it (list)}
    for source_ensembl, target_genes in combined_tf_to_genes.items():
        if target_ensembl in target_genes:
            source_symbol = ensembl_to_symbol.get(source_ensembl)
            if source_symbol and source_symbol in tf_exp.index and source_symbol != target_tf:
                predictor_tfs.append(source_symbol)

    # Need >= 2 predictor TFs for a valid LASSO run
    if len(predictor_tfs) < 2:
        continue

    # Feature matrix, shape is 550 patients * n predictor TFs (after transposing)
    X = tf_exp.loc[predictor_tfs].T.values

    # This is the response vector; it contains expression level of each TF in 550 patients 
    y = tf_exp.loc[target_tf].values

    # Standardize each prediction to mean = 0 and std = 1. 
    # This is to make coefficients for LASSO comparable so that large variance values don't dominate

    # If TF-A has expression values ranging from -0.1 to 0.1 across patients, and TF-B ranges from -10 to 10
    # then to explain the same amount of variance in the target TF's expression:
    # TF-A needs a large coefficient (e.g. 50) to produce meaningful changes in y
    # TF-B needs a small coefficient (e.g. 0.05) to produce the same effect

    # But since LASSO penalises large coeffs it treats A differently from B though both predict same signal
    X_scaled = StandardScaler().fit_transform(X)

    try:

        # Fits model using 5-fold cross-validation, 10k iters to ensure convergence and random_state to ensure reproducibility
        lasso = LassoCV(cv=5, max_iter=10000, random_state=42)
        lasso.fit(X_scaled, y)
        for j, coef in enumerate(lasso.coef_):

            # Appends all nonzero results to the dict
            if coef != 0:
                tf_tf_edges.append({
                    'Source': predictor_tfs[j],
                    'Target': target_tf,

                    # Sign convention, as mentioned above
                    'Edge':   1 if coef > 0 else -1,
                    'Coef':   coef
                })
    except:
        continue
        
# Convert to dataframe, generate overview and save
elapsed = time.time() - start
tf_network = pd.DataFrame(tf_tf_edges)
print(f"Completed in {elapsed/60:.1f} minutes")
print(f"TF-TF network edges: {len(tf_network)}")
print(f"Unique source TFs: {tf_network['Source'].nunique()}")
print(f"Unique target TFs: {tf_network['Target'].nunique()}")
tf_network.to_csv(os.path.join(matrices_dir, "tf_tf_network.csv"), index=False)
print("Saved.")

In [ ]:
# Compute outdegree/indegree ratio 
# outdegree: how many TFs this TF regulates
# indegree:  how many TFs regulate this TF
# out/in ratio: high ratio = master regulator (regulates many, regulated by few)
outdegree = tf_network.groupby('Source')['Target'].count().rename('outdegree')
indegree  = tf_network.groupby('Target')['Source'].count().rename('indegree')

# Replaces NaN values with 0
network_degrees = pd.DataFrame({'outdegree': outdegree, 'indegree': indegree}).fillna(0)

# Indegree plus 1 to prevent div by 0
network_degrees['out/in'] = network_degrees['outdegree'] / (network_degrees['indegree'] + 1)

# Creates a copy of index column with the name 'Alt_ID' because of MINER source code:
# network_degrees.loc[tf, "Alt_ID"]
network_degrees['Alt_ID'] = network_degrees.index

# Sort to get master regulators
network_degrees = network_degrees.sort_values('out/in', ascending=False)
print("Top 10 TFs by outdegree/indegree ratio:")
print(network_degrees.head(10))
network_degrees.to_csv(os.path.join(matrices_dir, "tf_network_degrees.csv"))

In [ ]:
# Build coverage dictionary: TF -> list of TFs it regulates
coverage_dict = {}
for tf in tf_network['Source'].unique():
    coverage_dict[tf] = tf_network[tf_network['Source'] == tf]['Target'].tolist()

In [ ]:
# Fixed infer_master_regulators (patches np.str_ bug in miner)
def infer_master_regulators_fixed(coverage_dict, network_degrees, unmix_tst):

    # Unmix returns clusters, which are groups of TFs coregulating the same targets
    # How? Find TF with most connections and take all TFs connected to it. Remove these; repeat iteratively until no TFs remain/ cluster is of size 1
    # From each cluster, pick the TFs with the most downstream targets
    max_coverage_tfs = [str(tf) for tf in select_optimal_tfs(unmix_tst, coverage_dict)]

    # Looks up the indegree and outdegree stats for each each TF in max_coverage_tfs; drops NaNs and sorts by out/in ratio
    selected_regulator_degrees = network_degrees.loc[
        [network_degrees.index[np.where(network_degrees.Alt_ID==tf)[0]][0]
         for tf in max_coverage_tfs if len(np.where(network_degrees.Alt_ID==tf)[0]) > 0], :]
    selected_regulator_degrees.dropna(inplace=True)
    selected_regulator_degrees.sort_values(by="out/in", ascending=False, inplace=True)

    # Adds TFs and measures the fraction of the network that they cover
    mapped_regs = []
    network_coverage = []

    # Since selected_regulator_degrees is sorted, we start with strongest master regulator first
    for i in range(selected_regulator_degrees.shape[0]):

        # Pick each TF
        key = str(selected_regulator_degrees.index[i])

        # Add TF's downstream targets to tmp_regs 
        tmp_regs = coverage_dict[key]

        # Set operations to ensure only unique values are added
        mapped_regs = union(mapped_regs, tmp_regs)

        # Network coverage definition
        network_coverage.append(len(mapped_regs) / float(network_degrees.shape[0]))

    # Stops adding TFs once marginal coverage gain < 5%
    tol = 0.05

    # Computes coverage difference
    diff_ = np.diff(network_coverage)

    cut = len(diff_)
    for j in range(len(diff_)):
        
        # Iterates through the succession of coverage differences
        if diff_[j] < tol:

            # Breaks loop as soon as it finds one less than tol; cut is set to this value of j
            cut = j
            break

    # Extracts the first cut values and outputs them as a list
    master_regulator_list = list(selected_regulator_degrees.index[0:cut+1])

    # Extracts final master regulators as a dict {TF: list of downstream TFS}
    master_regulator_dict = {tf: coverage_dict[str(network_degrees.loc[tf, "Alt_ID"])]
                             for tf in master_regulator_list}

    return master_regulator_list, master_regulator_dict, selected_regulator_degrees

In [ ]:
# Master regulators per state 
# For each state:
# 1. Find regulons differentially expressed in state patients vs rest (t-test)
# 2. Get TFs from causal flows that regulate those regulons
# 3. Build state-specific TF-TF subnetwork
# 4. Use unmix + select_optimal_tfs to find master regulators
# Note: tolerance cutoff removed to get broader coverage (paper: 67 master regulators)

state_master_regulators_all = {}

for state_id, state_patients in transcriptional_states.items():

    # For each state, split into 2 sets: patients in this state and other
    state_patients_set = set(state_patients)
    rest_patients      = list(set(eigengenes.columns) - state_patients_set)

    # Filters to patients with eigengene data
    state_pts_in_eig   = [p for p in state_patients if p in eigengenes.columns]

    # Skips if too few patients for a meaningful analysis
    if len(state_pts_in_eig) < 3 or len(rest_patients) < 3:
        continue

    # Vectorized t-test across all regulons

    # Eigengenes of state patients
    grp1   = eigengenes[state_pts_in_eig].values

    # Eigengenes of other patients
    grp2   = eigengenes[rest_patients].values

    # T-test, vectorized
    # T-test is a scaled difference of the mean of 2 datasets; larger the t-value, means datasets are likely to be different
    # p-value converts this to probability
    _, pvals = scipy_stats.ttest_ind(grp1, grp2, axis=1, equal_var=False)

    # List of regulons that are differentially active in this state
    # They pass the t-test i.e their mean expression in this state differs wrt other states
    state_regulons = eigengenes.index[pvals <= 0.05].astype(str).tolist()

    # TFs causally associated with state-specific regulons
    state_tfs = causal_results[
        causal_results['Regulon'].isin(state_regulons)
    ]['Regulator'].unique().tolist()

    # State-specific TFs, filtered to those actually in the causal network
    # This is because causal_results contains TFs that regulate regulons; we need the subset of this that regulates TFs
    state_tfs = [tf for tf in state_tfs if tf in coverage_dict]

    # Skips if too few TFs for a state
    if len(state_tfs) < 2:
        continue

    # State-specific TF-TF subnetwork
    state_out = {tf: [t for t in coverage_dict[tf] if t in state_tfs] for tf in state_tfs}
    
    # Dataframe with index, Alt_ID and outdeg and indeg info
    state_degrees = pd.DataFrame({
        'outdegree': [len(state_out[tf]) for tf in state_tfs],
        'indegree':  [sum(1 for targets in state_out.values() if tf in targets) for tf in state_tfs],
        'Alt_ID':    state_tfs
    }, index=state_tfs)

    # New column with ratio, for the TF-specific subnetwork as the ratio can differ
    state_degrees['out/in'] = state_degrees['outdegree'] / (state_degrees['indegree'] + 1)

    # Set index to Alt_ID; Alt_ID still exists as a column
    state_degrees.index = state_degrees['Alt_ID']

    # Binary matrix of TF-TF subnetwork that is input to unmix; the cell (i,j) is 1 if TF-i regulates TF-j and 0 otherwise
    coin_matrix = pd.DataFrame(0, index=state_tfs, columns=state_tfs)

    # state_out is a dict {TF: list of TFs it regulates}
    for tf, targets in state_out.items():
        for t in targets:
            if t in coin_matrix.columns:

                # Updates all interaction cells of the coin matrix to 1
                coin_matrix.loc[tf, t] = 1

    # Skips empty matrix (i.e no TF-TF edges)
    if coin_matrix.sum().sum() == 0:
        continue

    unmix_tst = unmix(coin_matrix)

    # All max_coverage_tfs ranked by out/in — no tolerance cutoff
    max_coverage_tfs = [str(tf) for tf in select_optimal_tfs(unmix_tst, state_out)]
    valid_tfs        = [tf for tf in max_coverage_tfs if tf in state_degrees.index]
    ranked           = state_degrees.loc[valid_tfs].sort_values('out/in', ascending=False)

    state_master_regulators_all[state_id] = ranked.index.tolist()
    print(f"State {state_id}: {len(ranked)} master regulators — {ranked.index[:5].tolist()}")

print(f"
Total states: {len(state_master_regulators_all)}")
all_masters = list(set([tf for tfs in state_master_regulators_all.values() for tf in tfs]))
print(f"Unique master regulators: {len(all_masters)}")
print(sorted(all_masters))

In [ ]:
# Convert to a final dataframe and save results 
master_reg_df = pd.DataFrame([
    {'State': state_id, 'MasterRegulator': tf}
    for state_id, tfs in state_master_regulators_all.items()
    for tf in tfs
])
master_reg_df.to_csv(os.path.join(matrices_dir, "master_regulators_per_state.csv"), index=False)
network_degrees.to_csv(os.path.join(matrices_dir, "tf_network_degrees.csv"))

print(f"Saved {len(master_reg_df)} master regulator-state associations")
print(f"Unique master regulators: {master_reg_df['MasterRegulator'].nunique()}")
print(f"States covered: {master_reg_df['State'].nunique()}")